<a href="https://colab.research.google.com/github/Ederson-Pinheiro/Projeto-SCTEC/blob/main/Mini_Projeto1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Contextualização e Desafio

Este notebook implementa um pipeline de sanitização de dados para os datasets de produtos e pedidos da Olist, utilizando apenas bibliotecas nativas do Python (`csv`, `re`, `datetime`). O objetivo é tratar inconsistências como dados ausentes, padronizar strings e aplicar regras de negócio específicas para garantir a qualidade dos dados para relatórios e modelos de Machine Learning.

 ## 1. Configuração e Importação de Bibliotecas

Primeiro, vamos importar as bibliotecas necessárias para as operações de arquivo, expressões regulares e manipulação de datas.

In [3]:
import csv
import re
from datetime import datetime
#import requests # Para baixar os arquivos do GitHub
#import os # Para manipular arquivos

 ## 2. Download dos Datasets

BAIXANDO DATASETS (WITH OPEN)

In [4]:
with open("/content/drive/MyDrive/olist_orders_dataset.csv", "r", newline='', encoding="utf-8") as arquivo:
    leitor = csv.DictReader(arquivo)
    pedidos = list(leitor)

with open("/content/drive/MyDrive/olist_products_dataset.csv", "r", newline='', encoding="utf-8") as arquivo:
    leitor = csv.DictReader(arquivo)
    produtos = list(leitor)


Vamos baixar os arquivos olist_products_dataset.csv e olist_orders_dataset.csv diretamente do repositório do GitHub fornecido.

In [5]:
# def baixando_arquivo(url, local_arquivo):
#     """Baixa um arquivo de uma URL e salva localmente."""
#     print(f"Baixando {local_arquivo}...")
#     with requests.get(url, stream=True) as r:
#         r.raise_for_status()
#         with open(local_arquivo, 'wb') as f:
#             for chunk in r.iter_content(chunk_size=8192):
#                 f.write(chunk)
#     print(f"Download de {local_arquivo} concluído.")

# # URLs dos arquivos brutos no GitHub
# produtos_url = 'https://raw.githubusercontent.com/fiesc-junior-prado/mine_projeto_bloco_1/main/olist_products_dataset.csv'
# pedidos_url = 'https://raw.githubusercontent.com/fiesc-junior-prado/mine_projeto_bloco_1/main/olist_orders_dataset.csv'

# # Nomes dos arquivos locais
# arquivo_produtos = 'olist_products_dataset.csv'
# arquivo_pedidos = 'olist_orders_dataset.csv'

# # Executar download
# baixando_arquivo(produtos_url, arquivo_produtos)
# baixando_arquivo(pedidos_url, arquivo_pedidos)

 ## 3. Validação e Tratamento de Dados Ausentes, Padronização de Strings e Regex (Dataset de Produtos)

Esta etapa irá processar o `olist_products_dataset.csv` para:
- Preencher `product_category_name` nulo/vazio com "Sem Categoria".
- Tratar valores nulos nas dimensões físicas (`product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm`) Em vez de substituir dimensões nulas pela "média", escolhi usar a mediana e preencher com ela. Isso costuma ser mais adequado para análises e modelos de Machine Learning, porque evita outliers.
- Converter nomes de categorias para minúsculas e remover espaços em branco excedentes.
- Limpar caracteres especiais/pontuações indevidas usando Expressões Regulares.

In [6]:
from statistics import median

def calcular_medianas(produtos_data):

    colunas_dimensoes = [
        'product_weight_g',
        'product_length_cm',
        'product_height_cm',
        'product_width_cm'
    ]

    medianas = {}

    for coluna in colunas_dimensoes:

        valores = [
            float(produto[coluna])
            for produto in produtos_data
            if produto.get(coluna, '').strip() != ''
        ]

        medianas[coluna] = (
            median(valores)
            if valores else 0
        )

    return medianas

def processar_produtos(produtos_data):

    medianas = calcular_medianas(produtos_data)

    produtos_sanitizados = []

    total_produtos_processados = 0
    categorias_nulas_corrigidas = 0
    dimensoes_nulas_corrigidas = 0

    colunas_dimensoes = [
        'product_weight_g',
        'product_length_cm',
        'product_height_cm',
        'product_width_cm'
    ]

    for produto_original in produtos_data:

        # Crie uma cópia para evitar modificar a lista original no mesmo local
        produto = dict(produto_original)
        total_produtos_processados += 1

        # Categoria nula
        categoria = produto.get(
            'product_category_name',
            ''
        )

        if categoria.strip() == '':
            produto['product_category_name'] = (
                'Sem Categoria'
            )
            categorias_nulas_corrigidas += 1

        # Dimensões nulas
        for coluna in colunas_dimensoes:

            valor = produto.get(
                coluna,
                ''
            ).strip()

            if valor == '':
                produto[coluna] = str(
                    round(medianas[coluna], 2)
                )
                dimensoes_nulas_corrigidas += 1

        # Padronização de categoria(lower, strip e regex)
        categoria_limpa = (
            produto['product_category_name']
            .lower()
            .strip()
        )

        categoria_limpa = re.sub(
            r'[^a-z0-9\s]',
            '',
            categoria_limpa
        )

        categoria_limpa = re.sub(
            r'\s+',
            ' ',
            categoria_limpa
        ).strip()

        produto[
            'product_category_name_sanitized'
        ] = categoria_limpa

        produtos_sanitizados.append(
            produto
        )

    return (
        produtos_sanitizados,
        total_produtos_processados,
        categorias_nulas_corrigidas,
        dimensoes_nulas_corrigidas
    )

## 4. Lógica de Regra de Negócio e Formatação Temporal (olist_orders_dataset.csv)

Nesta seção, trataremos o arquivo `olist_orders_dataset.csv`:
1. **Regra de Negócio:** Validar a hipótese de que `order_delivered_customer_date` está nulo quando o `order_status` é 'canceled'.
2. **Formatação Temporal:** Converter `order_approved_at` para o formato de data simplificado brasileiro (DD/MM/YYYY).

In [7]:
def processar_pedidos(pedidos_data):

    pedidos_sanitizados = []

    total_pedidos_processados = 0
    pedidos_data_entrega_nula = 0
    pedidos_cancelados_com_data_entrega_nula = 0
    pedidos_cancelados_outros = 0
    datas_formatadas = 0

    for pedido_original in pedidos_data:

        # Crie uma cópia para evitar modificar a lista original no mesmo local
        pedido = dict(pedido_original)
        total_pedidos_processados += 1

        # ----------------------------------------
        # Regra de negócio
        # ----------------------------------------

        entrega = pedido.get(
            'order_delivered_customer_date',
            ''
        )

        status = pedido.get(
            'order_status',
            ''
        )

        if entrega.strip() == '':

            pedidos_data_entrega_nula += 1

            if status == 'canceled':
                pedidos_cancelados_com_data_entrega_nula += 1

        elif status == 'canceled':

            pedidos_cancelados_outros += 1

        # ----------------------------------------
        # Formatação temporal (DATETIME)
        # ----------------------------------------

        data_aprovacao = pedido.get(
            'order_approved_at',
            ''
        )

        if data_aprovacao.strip() != '':

            try:

                data_obj = datetime.strptime(
                    data_aprovacao,
                    '%Y-%m-%d %H:%M:%S'
                )

                pedido[
                    'order_approved_at_formatted'
                ] = data_obj.strftime(
                    '%d/%m/%Y'
                )

                datas_formatadas += 1

            except ValueError:

                pedido[
                    'order_approved_at_formatted'
                ] = 'Data Inválida'

        else:

            pedido[
                'order_approved_at_formatted'
            ] = 'Data Ausente'

        pedidos_sanitizados.append(
            pedido
        )

    return (
        pedidos_sanitizados,
        total_pedidos_processados,
        pedidos_data_entrega_nula,
        pedidos_cancelados_com_data_entrega_nula,
        pedidos_cancelados_outros,
        datas_formatadas
    )

 ## 5. Relatório de Status Manual e Execução Principal

Esta seção executa as funções de processamento e gera um sumário estatístico final, conforme os requisitos do projeto.

In [8]:
# --- Execução do Pipeline ---

print("\n--- Iniciando o Pipeline de Sanitização de Dados ---")

# Processar produtos
(
    produtos_processados_data,
    total_produtos_proc,
    cat_nulas_corr,
    dim_nulas_corr
) = processar_produtos(produtos)

print(f"\nTotal de linhas de produtos processadas: {total_produtos_proc}")
print(f"Categorias de produtos nulas corrigidas para 'Sem Categoria': {cat_nulas_corr}")
print(f"Dimensões físicas de produtos nulas/vazias corrigidas pela média da coluna: {dim_nulas_corr}")

# Processar pedidos
(
    pedidos_processados_data,
    total_pedidos_proc,
    data_entrega_nula,
    cancelados_com_data_nula,
    cancelados_outros,
    datas_formatadas
) = processar_pedidos(pedidos)

print(f"\nTotal de linhas de pedidos processadas: {total_pedidos_proc}")
print(f"Pedidos com 'order_delivered_customer_date' nula/vazia: {data_entrega_nula}")
print(f"Pedidos cancelados com 'order_delivered_customer_date' nula/vazia: {cancelados_com_data_nula}")
print(f"Total de datas de aprovação formatadas: {datas_formatadas}")

# ----------------------------------------
# Verificação da hipótese de negócio
# ----------------------------------------

if data_entrega_nula == cancelados_com_data_nula:

    print(
        "\nHipótese de Negócio Olist COMPROVADA: "
        "Todos os pedidos com data de entrega nula "
        "possuem status 'canceled'."
    )

else:

    print(
        "\nHipótese de Negócio Olist NÃO COMPROVADA: "
        "Existem pedidos com data de entrega nula "
        "e status diferente de 'canceled'."
    )

    print(
        "Pedidos com data de entrega nula e "
        f"status diferente de 'canceled': "
        f"{data_entrega_nula - cancelados_com_data_nula}"
    )

print(
    f"Pedidos cancelados com data de entrega preenchida: "
    f"{cancelados_outros}"
)

# ----------------------------------------
# Sumário Estatístico
# ----------------------------------------

print("\n--- SUMÁRIO ESTATÍSTICO FINAL ---")

print(f"Total de produtos processados: {total_produtos_proc}")
print(f"Total de pedidos processados: {total_pedidos_proc}")

print(
    f"Categorias de produtos corrigidas: "
    f"{cat_nulas_corr}"
)

print(
    f"Dimensões físicas corrigidas pela média: "
    f"{dim_nulas_corr}"
)

print(
    f"Datas de aprovação formatadas: "
    f"{datas_formatadas}"
)

print(
    f"Pedidos cancelados com entrega nula: "
    f"{cancelados_com_data_nula}"
)

print(
    f"Inconsistências encontradas "
    f"(cancelado com entrega): "
    f"{cancelados_outros}"
)

# ----------------------------------------
# Exemplos dos dados tratados
# ----------------------------------------

print(
    "\n--- Exemplo de Produtos Sanitizados "
    "(primeiras 5 linhas) ---"
)

for i, produto in enumerate(produtos_processados_data[:5]):

    print(
        f"Produto {i+1}: "
        f"Categoria = '{produto.get('product_category_name')}' | "
        f"Sanitizada = '{produto.get('product_category_name_sanitized')}' | "
        f"Peso = '{produto.get('product_weight_g')}'"
    )

print(
    "\n--- Exemplo de Pedidos Sanitizados "
    "(primeiras 5 linhas) ---"
)

for i, pedido in enumerate(pedidos_processados_data[:5]):

    print(
        f"Pedido {i+1}: "
        f"Aprovado = '{pedido.get('order_approved_at')}' | "
        f"Formatado = '{pedido.get('order_approved_at_formatted')}' | "
        f"Entrega = '{pedido.get('order_delivered_customer_date')}' | "
        f"Status = '{pedido.get('order_status')}'"
    )


--- Iniciando o Pipeline de Sanitização de Dados ---

Total de linhas de produtos processadas: 32951
Categorias de produtos nulas corrigidas para 'Sem Categoria': 610
Dimensões físicas de produtos nulas/vazias corrigidas pela média da coluna: 8

Total de linhas de pedidos processadas: 99441
Pedidos com 'order_delivered_customer_date' nula/vazia: 2965
Pedidos cancelados com 'order_delivered_customer_date' nula/vazia: 619
Total de datas de aprovação formatadas: 99281

Hipótese de Negócio Olist NÃO COMPROVADA: Existem pedidos com data de entrega nula e status diferente de 'canceled'.
Pedidos com data de entrega nula e status diferente de 'canceled': 2346
Pedidos cancelados com data de entrega preenchida: 6

--- SUMÁRIO ESTATÍSTICO FINAL ---
Total de produtos processados: 32951
Total de pedidos processados: 99441
Categorias de produtos corrigidas: 610
Dimensões físicas corrigidas pela média: 8
Datas de aprovação formatadas: 99281
Pedidos cancelados com entrega nula: 619
Inconsistências en

##RELATÓRIO

In [9]:
entrega_cancelado = cancelados_com_data_nula
entrega_nao_cancelado = data_entrega_nula - cancelados_com_data_nula
total_cancelados = cancelados_com_data_nula + cancelados_outros
categorias_corrigidas = cat_nulas_corr
dimensoes_corrigidas = dim_nulas_corr
datas_convertidas = datas_formatadas

resultado_hipotese = (
    "Hipótese confirmada."
    if entrega_nao_cancelado == 0
    else "Hipótese rejeitada."
)

with open(
    "relatorio_sanitizacao_olist.txt",
    "w",
    encoding="utf-8"
) as arquivo:

    arquivo.write(
        f"===== RELATÓRIO DE SANITIZAÇÃO DE DADOS OLIST =====\n\n"

        f"FASE 1 - TRATAMENTO DE DADOS AUSENTES\n\n"
        f"Total de produtos processados: {len(produtos)}\n"
        f"Categorias corrigidas: {categorias_corrigidas}\n"
        f"Dimensões corrigidas pela média: {dimensoes_corrigidas}\n\n"

        f"FASE 2 - PADRONIZAÇÃO DE CATEGORIAS\n\n"
        f"Categorias convertidas para minúsculas e "
        f"caracteres especiais removidos.\n\n"

        f"FASE 3 - VALIDAÇÃO DA REGRA DE NEGÓCIO\n\n"
        f"Entrega nula e cancelado: {entrega_cancelado}\n"
        f"Entrega nula e não cancelado: {entrega_nao_cancelado}\n"
        f"{resultado_hipotese}\n\n"

        f"FASE 4 - FORMATAÇÃO TEMPORAL\n\n"
        f"Datas convertidas: {datas_convertidas}\n\n"

        f"FASE 5 - RESUMO FINAL\n\n"
        f"Pedidos processados: {len(pedidos)}\n"
        f"Pedidos cancelados: {total_cancelados}\n\n"

        f"Base sanitizada com sucesso."
    )

print(
    "\nRelatório 'relatorio_sanitizacao_olist.txt' "
    "gerado com sucesso!"
)



Relatório 'relatorio_sanitizacao_olist.txt' gerado com sucesso!
